# Row-wise Δ on the live MCP agent

Same setting as `Live_agent_w_baselines.ipynb`: a 3B model picks the MCP servers of a GPT-4.1-mini agent, and the reward is
whether the agent then finished the task.  What changes here is the control variate.  `Q + Δ` uses one scalar per
(prefix, token) and removes 41% of the noise the Q table leaves.  `Q + rowΔ` gives the residual one scalar per
(prefix, token, **weight row**) instead.  That needs no linear solve and removes 67%.

In [ ]:
import os; os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import json, re, copy, itertools, collections
from types import SimpleNamespace
import numpy as np, torch, matplotlib.pyplot as plt
from dotenv import load_dotenv; load_dotenv(".env")         # HF_TOKEN (Llama is gated); OPENAI_API_KEY only to re-run the agent
from mcp_live import workspace, catalog, tasks as task_lib, agent

STRICT = True       # reward also requires that every attached server was used; False: the checker alone
FREE_TEXT = True       # the model writes the whole answer; False: only the four digits are sampled (constrained decoding)

SLOTS, BUDGET = catalog.SLOTS, catalog.BUDGET                    # 4 slots x 3 servers; the client attaches at most 2
CONFIGS = list(itertools.product(range(4), repeat=4))            # one digit per slot, 0 = none: 256 configurations
OVER_CAP = np.array([sum(d > 0 for d in c) > BUDGET for c in CONFIGS])
servers_of = lambda c: [SLOTS[j][1][d - 1] for j, d in enumerate(c) if d > 0]

TASKS = {t["id"]: t for t in task_lib.make_tasks(workspace.build_template(), workspace.start_intranet())}
SERVERS = json.load(open("mcp_live/catalog.json"))              # what each server announces about itself over MCP
for slot, names in SLOTS:
    print(f"{slot:<10}", " | ".join(names))
print()
for t in TASKS.values():
    print(f"{t['id']:<10} {t['spec'][:110]}")

In [ ]:
RUN_AGENT = False
EPISODES_FILE = "_live_rewards_gpt-4.1-mini_transcripts.jsonl"
if RUN_AGENT:
    EPISODES_FILE = "my_episodes.jsonl"
    await agent.fill_cache(list(TASKS.values()), CONFIGS, "my_rewards.json", repeats=2, transcripts_path=EPISODES_FILE)

episodes = [json.loads(line) for line in open(EPISODES_FILE)]

def reward(ep):                                                  # one episode -> 0 or 1
    used = {call.split("__")[0] for call in ep["calls"]}         # the servers the agent actually called
    return float(ep["f"] and (not STRICT or set(ep["servers"]) <= used))

runs = collections.defaultdict(list)
for ep in episodes:
    runs[ep["task"], tuple(ep["config"])].append(reward(ep))
REWARD = {t: np.array([np.mean(runs.get((t, c), [0.0])) for c in CONFIGS]) for t in TASKS}   # over the cap: never run, 0

for t, r in REWARD.items():
    wins = [CONFIGS[i] for i in np.flatnonzero(r > 0)]
    print(f"{t:<10} {len(wins):>2} of 256 succeed: " + ", ".join("+".join(servers_of(c)) for c in wins[:5]) + (" ..." if len(wins) > 5 else ""))

In [ ]:
ep = next(e for e in episodes if e["task"] == "port" and e["servers"] == ["filesystem", "mail"])
for line in ep["transcript"]:
    print(line.replace("\n", " ")[:130])
print(f"\nchecker: {ep['f']}   servers given: {ep['servers']}   reward: {reward(ep)}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, clean_up_tokenization_spaces=False)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).cuda().requires_grad_(False)
torch.manual_seed(0)                                             # lora_A's random draw sets the gradient geometry: fix it
model = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["down_proj"], layers_to_transform=[27])).eval()
params = [p for n, p in model.named_parameters() if "lora" in n]
start_params = [p.detach().clone() for p in params]
def reset():
    for p, p0 in zip(params, start_params): p.data.copy_(p0)

SYSTEM = ("You configure the MCP servers of an autonomous agent that will carry out a small task in a project workspace. "
          "There are four capability slots; each offers three servers, and 0 leaves the slot empty. The client attaches at most "
          f"{BUDGET} servers in total: a configuration with more than {BUDGET} is rejected and the task fails. Attach exactly the "
          "servers the task needs: the agent can only use what you attach, and a missing or wrong server makes the task "
          "impossible. Answer with one line per slot, in the order given, in exactly this form:\n"
          + "\n".join(f"{slot}: <0|1|2|3>" for slot, _ in SLOTS) + "\nEND")

def legend(slot, names):
    return f"{slot}: 0 = no server; " + "; ".join(
        f"{i + 1} = {n} ({SERVERS[n]['instructions'].rstrip('.')}; tools: {', '.join(t['name'] for t in SERVERS[n]['tools'])})"
        for i, n in enumerate(names))

PREFILL = "files: "
enc = lambda s: tok.encode(s, add_special_tokens=False)
def prompt_ids(task_id):
    user = f"Task: {TASKS[task_id]['spec']}\nSlots:\n" + "\n".join(legend(s, n) for s, n in SLOTS)
    chat = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    return enc(chat + PREFILL)

DIGITS = [enc(str(d))[0] for d in range(4)]
TEMPLATE = []
for j, (slot, _) in enumerate(SLOTS):
    TEMPLATE += (enc(f"\n{slot}: ") if j else []) + [None]
fill = lambda template, digits: [DIGITS[digits[template[:i].count(None)]] if t is None else t for i, t in enumerate(template)]
assert enc(PREFILL + "1\nknowledge: 0\nexecution: 2\ndelivery: 3") == enc(PREFILL) + fill(TEMPLATE, (1, 0, 2, 3))
print("answer template:", PREFILL + "".join("<d>" if t is None else tok.decode([t]) for t in TEMPLATE).replace("\n", "\\n"))

What the untrained model writes on its own:

In [ ]:
ANSWER = re.compile(r"files: ([0-3])\nknowledge: ([0-3])\nexecution: ([0-3])\ndelivery: ([0-3])(?!\d)")
def parse(text):
    m = ANSWER.match(text); return CONFIGS.index(tuple(map(int, m.groups()))) if m else -1

job = "lint"
ids = torch.tensor([prompt_ids(job)] * 6, device="cuda")
torch.manual_seed(0)
out = model.generate(ids, attention_mask=torch.ones_like(ids), do_sample=True, temperature=1.0, top_p=1.0, top_k=0,     # the full distribution (Llama's defaults truncate it)
                     max_new_tokens=24, pad_token_id=tok.eos_token_id)
for row in out:
    text = PREFILL + tok.decode(row[ids.shape[1]:], skip_special_tokens=True); c = parse(text)
    got = ("+".join(servers_of(CONFIGS[c])) or "nothing") if c >= 0 else "unparseable"
    print(f"{text[:70]!r:<74} -> {got:<22} reward {REWARD[job][c] if c >= 0 else 0.0}")

In [ ]:
lm, last = model.base_model.model, model.base_model.model.model.layers[-1]
dp = last.mlp.down_proj                                          # the one layer that is trained
LA, LB, SC = dp.lora_A.default.weight, dp.lora_B.default.weight, dp.scaling["default"]
saved = {}
last.post_attention_layernorm.register_forward_pre_hook(lambda m, a: saved.update(r=a[0]))    # the residual stream
dp.register_forward_pre_hook(lambda m, a: saved.update(x=a[0]))                               # the LoRA's input

def build_tree(task_id, chunk=64):
    prompt = prompt_ids(task_id)
    steps = [k for k, t in enumerate(TEMPLATE) if t is None or FREE_TEXT]                   # positions the model is scored on
    nodes = [(k, d) for k in steps for d in itertools.product(range(4), repeat=TEMPLATE[:k].count(None))]
    allowed = [DIGITS if TEMPLATE[k] is None else [TEMPLATE[k]] for k, _ in nodes]         # plus one branch: anything else
    size = [len(a) + 1 for a in allowed]; first = np.cumsum([0] + size[:-1]); E = sum(size)

    leaves = []                                                  # (entries taken on the way, configuration index or -1)
    def walk(i, digits, path):
        n = nodes.index((steps[i], digits)); A = len(allowed[n])
        leaves.append((path + [first[n] + A], -1))                                          # anything else
        for j in range(A):
            nxt = digits + (j,) if A == 4 else digits
            if i + 1 < len(steps): walk(i + 1, nxt, path + [first[n] + j])
            else: leaves.append((path + [first[n] + j], CONFIGS.index(nxt)))
    walk(0, (), [])
    node_first, node_size = np.repeat(first, size), np.repeat(size, size)
    taken, passed = np.zeros((len(leaves), E)), np.zeros((len(leaves), E))                 # passed: every branch of every node on the way
    for y, (path, _) in enumerate(leaves):
        taken[y, path] = 1
        for e in path: passed[y, node_first[e]:node_first[e] + node_size[e]] = 1

    # One forward: the 64 answers that spell out every choice of the first three digits contain every prefix in the tree.
    lines = list(itertools.product(range(4), repeat=TEMPLATE.count(None) - 1))
    with torch.no_grad():
        kv = lm.model(input_ids=torch.tensor([prompt[:-1]], device="cuda")).past_key_values; kv.batch_repeat_interleave(len(lines))
        lm.model(input_ids=torch.tensor([prompt[-1:] + fill(TEMPLATE[:-1], d) for d in lines], device="cuda"), past_key_values=kv)
    X, R = saved["x"], saved["r"]                                                          # (64, 16, .): position k predicts TEMPLATE[k]
    at = [(lines.index(d + (0,) * (len(lines[0]) - len(d))), k) for k, d in nodes]

    P = torch.zeros(E, dtype=torch.float64, device="cuda"); Z = torch.zeros(E, sum(p.numel() for p in params), device="cuda")
    U = torch.zeros(E, R.shape[-1], device="cuda")                                         # the backprop signal behind every score
    W = lm.lm_head.weight
    for A1 in sorted(set(size)):
        group = [n for n in range(len(nodes)) if size[n] == A1]
        for c in range(0, len(group), chunk):
            g = group[c:c + chunk]; B = len(g); s, k = map(list, zip(*[at[n] for n in g]))
            v = R[s, k][:, None] + dp(X[s, k][:, None])                                    # the vector the LoRA writes into
            h = lm.model.norm(v)[:, 0]                                                     # (B, d), carries the LoRA gradient
            with torch.no_grad():
                logp = torch.log_softmax(lm.lm_head(h), -1); p = logp.exp()
                ok = torch.tensor([allowed[n] for n in g], device="cuda")
                rest = torch.ones_like(p).scatter_(1, ok, 0.0)                                # the tokens that break the answer
                lp = torch.cat([logp.gather(1, ok), (logp + rest.log()).logsumexp(-1, keepdim=True)], 1)
                q = p * rest / (p * rest).sum(-1, keepdim=True).clamp_min(1e-30)
                dlp_dh = torch.cat([W[ok], (q @ W)[:, None]], 1) - (p @ W)[:, None]          # d log p(branch) / d h, (B, A1, d)
            onehot = torch.zeros(B * A1, B, h.shape[1], device="cuda")
            onehot[torch.arange(B * A1), torch.arange(B).repeat_interleave(A1)] = dlp_dh.reshape(B * A1, -1)
            gs = torch.autograd.grad(h, [v] + params, onehot, is_grads_batched=True)         # one score per branch, and the signal behind it
            rows = torch.tensor([first[n] + a for n in g for a in range(A1)], device="cuda")
            pr = lp.double().exp(); P[rows] = (pr / pr.sum(1, keepdim=True)).flatten()
            Z[rows] = torch.cat([x.reshape(B * A1, -1) for x in gs[1:]], 1)
            U[rows] = gs[0][torch.arange(B * A1), torch.arange(B).repeat_interleave(A1), 0]
            del onehot, gs

    T = lambda a: torch.tensor(a, dtype=torch.float64, device="cuda")
    node = torch.tensor(np.repeat(np.arange(len(nodes)), size), device="cuda")              # the node of every entry
    tree = SimpleNamespace(P=P, Z=Z, K=(Z @ Z.T).double(), taken=T(taken), passed=T(passed), N=len(nodes), node=node,
                           conf=np.array([c for _, c in leaves]))
    tree.pb = torch.exp(tree.taken @ torch.log(P.clamp_min(1e-300)))                        # probability of every leaf

    # The row structure.  A linear layer's gradient is (backprop signal) x (layer input), so the score of branch e at prefix n is
    # gA[e] (x) x_n on lora_A's rows and u[e] (x) z_n on lora_B's rows: inside one row every branch of n writes along the SAME vector.
    with torch.no_grad():
        s_at, k_at = map(list, zip(*at)); xn = X[s_at, k_at].double(); zn = SC * (xn @ LA.double().T)
        tree.gA, tree.gB = SC * (U.double() @ LB.double()), U.double()
        tree.KA, tree.KB = tree.gA @ tree.gA.T, tree.gB @ tree.gB.T                         # <gA_e, gA_e'> and <u_e, u_e'>
        tree.XX, tree.ZZ = (xn @ xn.T)[node][:, node], (zn @ zn.T)[node][:, node]           # <x_n, x_n'> and <z_n, z_n'>, per pair of entries
        tree.nx, tree.nz = tree.XX.diagonal().sqrt(), tree.ZZ.diagonal().sqrt()
        tree.xhat, tree.zhat = xn / xn.norm(dim=1, keepdim=True), zn / zn.norm(dim=1, keepdim=True)
    return tree

def leaf_rewards(tree, table):                                   # a table over the 256 configurations -> one reward per leaf
    return torch.tensor(np.where(tree.conf >= 0, table[np.maximum(tree.conf, 0)], 0.0), device="cuda")

tree = build_tree(job); f = leaf_rewards(tree, REWARD[job])
print(f"{job}: {tree.P.numel()} (prefix, token) entries, {len(tree.conf)} leaves;  P(parseable) = {tree.pb[tree.conf >= 0].sum():.3f},  P(success) = {tree.pb @ f:.4f}")

### A residual with one scalar per weight row

Write the Q estimator as a sum over the prefixes of the answer.  At prefix $n$ the branch $v$ that was drawn contributes the
martingale increment $\xi_v = A_v S_n + D_v - \bar D_n$ ($A_v = Q_v - V_n$, $S_n$ the summed score of the path so far,
$D$ the expected local gradients below), and $\sum_v P_v \xi_v = 0$, so $\operatorname{Var}[\hat g_Q] = \sum_n \mathbb{E}\,\|\xi\|^2$.

A scalar $\Delta$ subtracts a multiple of *that branch's own score*, so it can only cancel the part of $\xi_v$ that points along
$s_n(v)$.  With four digits to pick from, the scores at a prefix point in different directions and little of $\xi_v$ lies along
any one of them: $\Delta$ removes 41% of what $Q$ leaves.

A linear layer's gradient is (backprop signal) $\otimes$ (layer input), so **inside one weight row every branch's score is a
multiple of the same vector**: $z_n = \mathrm{SC}\cdot A x_n$ for the rows of `lora_B`, $x_n$ for the rows of `lora_A`.  Give the
residual one scalar per (prefix, token, row) and it cancels, row by row, the whole component of $\xi_v$ along that vector:

$$\hat g \;=\; \hat g_Q - \sum_{n \text{ on the path}} \Pi_n\, \xi_{v_n}, \qquad \Pi_n = \text{project every row on the layer input at } n.$$

$\Pi_n$ is linear and $\sum_v P_v \xi_v = 0$, so the residual is mean zero at every prefix: the estimator stays unbiased for any
reward table, and there is no linear solve.  The scalar $\Delta$ is the special case where all rows share one scalar, so
`Q + rowΔ` can never be worse than `Q + Δ`.

In [ ]:
def weights(tree, c, f):             # g_hat(y) = weights[y] @ Z, for every leaf y at once
    return tree.taken * (f[:, None] - c) + tree.passed * (tree.P * c)

def noise(tree, c, f):               # exact Var[g_hat] under the current policy
    A = weights(tree, c, f); D = A - tree.pb @ A
    return float(tree.pb @ ((D @ tree.K) * D).sum(1))

def Q_table(tree, f):                # E[f | the answer reached this prefix and wrote this token]
    n = tree.pb @ tree.taken
    return torch.where(n > 0, (tree.pb * f) @ tree.taken / n.clamp_min(1e-300), 0.0)

def Delta_table(tree, f, Q):         # argmin over Δ of Var[g_hat] with c = Q + Δ: one scalar per (prefix, token)
    A = weights(tree, Q, f); R = A - tree.pb @ A
    dA = tree.passed * tree.P - tree.taken                      # weights(c + Δ) = weights(c) + dA * Δ
    M = tree.K * (dA.T @ (tree.pb[:, None] * dA)); b = -(((R @ tree.K) * dA).T @ tree.pb)
    return torch.linalg.pinv(M, hermitian=True, rtol=1e-8) @ b

# ---- the row-wise residual: no solve, just the increments and a projection ----------------------------------------------
def increments(tree, c, f):
    """ξ, in weight space: how the estimator's conditional mean moves when the answer takes branch e at its prefix.
    A row of ξ is read like a row of weights(), so the increment itself is ξ[e] @ Z."""
    A = weights(tree, c, f); n = tree.pb @ tree.taken                                        # n: P(the answer takes e)
    cond = ((tree.taken * tree.pb[:, None]).T @ A) / n.clamp_min(1e-300)[:, None]            # E[weights | e taken]
    mean = torch.zeros(tree.N, A.shape[1], dtype=A.dtype, device=A.device).index_add_(0, tree.node, tree.P[:, None] * cond)
    return cond - mean[tree.node]                                                            # zero mean over the branches of every prefix

def row_parts(tree, xi, e=slice(None)):
    """the part of the increments a row-wise residual cancels: their component along the layer input, row by row.
    Rows of lora_A see <x_n, .>, rows of lora_B see <z_n, .>; the other factor of each score is carried by gA / gB."""
    return xi * tree.XX[e] / tree.nx[e][:, None], xi * tree.ZZ[e] / tree.nz[e][:, None]

def noise_row(tree, c, f, xi):       # exact Var[g_hat] once the residual built from the increments xi is subtracted
    x = increments(tree, c, f); left = ((x @ tree.K) * x).sum(1)
    for a, b, K in zip(row_parts(tree, x), row_parts(tree, xi), (tree.KA, tree.KB)):
        left = left + ((b @ K) * b).sum(1) - 2 * ((a @ K) * b).sum(1)                        # |ξ - Π ξ_hat|^2, row by row
    return float((tree.pb @ tree.taken) @ left)

def row_correction(tree, xi, y):     # what one episode subtracts: Π_n ξ at every prefix on its path, as a flat gradient
    e = tree.taken[y].nonzero().squeeze(1); n = tree.node[e]
    cA, cB = row_parts(tree, xi[e], e)
    return torch.cat([torch.einsum("eq,ej->qj", cA @ tree.gA, tree.xhat[n]).flatten(),
                      torch.einsum("ei,eq->iq", cB @ tree.gB, tree.zhat[n]).flatten()])

class Seen:
    # the agent episodes so far, as f-hat per configuration
    def __init__(self):
        self.n, self.sum = np.zeros(len(CONFIGS)), np.zeros(len(CONFIGS))
    def add(self, conf, f):
        if conf >= 0: self.n[conf] += 1; self.sum[conf] += f
    def f_hat(self, prior=1.0):
        tried = (self.n > 0) & ~OVER_CAP
        m = (self.sum[tried] / self.n[tried]).mean() if tried.any() else 0.0
        return np.where(OVER_CAP, 0.0, (self.sum + prior * m) / (self.n + prior))   # over the cap: the client's rule, a known 0

def control_variate(method, tree, seen, f_true):
    """the table c, and for the row-wise methods the increments its residual is built from (None otherwise)."""
    if method == "REINFORCE":
        return torch.zeros_like(tree.P), None                                # plain REINFORCE: no baseline at all
    f = f_true if "exact" in method else leaf_rewards(tree, seen.f_hat())     # "(exact)": the true reward, a ceiling
    Q = Q_table(tree, f)
    if "rowΔ" in method: return Q, increments(tree, Q, f)
    if "Δ" in method: return Q + Delta_table(tree, f, Q), None
    return Q, None

def estimator(tree, c, xi, f, ys):   # the gradient of a batch of episodes, and the exact noise of the estimator behind it
    g = weights(tree, c, f)[ys].mean(0).float() @ tree.Z
    if xi is None: return g, noise(tree, c, f)
    return g - torch.stack([row_correction(tree, xi, y) for y in ys]).mean(0).float(), noise_row(tree, c, f, xi)

def sample(tree, n, rng):
    return rng.choice(len(tree.conf), size=n, p=(tree.pb / tree.pb.sum()).cpu().numpy())

In [ ]:
Q = Q_table(tree, f); xi = increments(tree, Q, f); w = tree.pb @ tree.taken                  # w: P(the answer takes e)
rf = noise(tree, torch.full_like(tree.P, float(tree.pb @ f)), f); nq = noise(tree, Q, f); nr = noise_row(tree, Q, f, xi)

print(f"every score is (signal) x (input):   max|K - (KA*XX + KB*ZZ)| / max|K| = "
      f"{float((tree.KA * tree.XX + tree.KB * tree.ZZ - tree.K).abs().max() / tree.K.abs().max()):.1e}")
print(f"the increments are the whole noise:  sum_e P(take e) |xi_e Z|^2 / Var[g_Q] = {float(w @ ((xi @ tree.K) * xi).sum(1)) / nq:.4f}")
print(f"the residual has mean zero:          max |E[residual]| over lora_B's entries = "
      f"{float(torch.einsum('e,ei,eq->iq', w, row_parts(tree, xi)[1] @ tree.gB, tree.zhat[tree.node]).abs().max()):.1e}")
print(f"\n{job}:  REINFORCE 10.00    Q {10 * nq / rf:.2f}    Q + Δ {10 * noise(tree, Q + Delta_table(tree, f, Q), f) / rf:.2f}"
      f"    Q + rowΔ {10 * nr / rf:.2f}")

rng = np.random.default_rng(0); A = weights(tree, Q, f); gbar = (tree.pb @ A).float() @ tree.Z     # the true gradient
ys = sample(tree, 2000, rng); mean, vq, vr = 0.0, 0.0, 0.0
for i in range(0, len(ys), 200):                                 # Monte Carlo, one episode at a time, exactly as in training
    ch = ys[i:i + 200]; G = A[ch].float() @ tree.Z; C = torch.stack([row_correction(tree, xi, y) for y in ch]).float()
    mean = mean + (G - C).sum(0); vq += float(((G - gbar) ** 2).sum()); vr += float(((G - C - gbar) ** 2).sum())
print(f"{len(ys)} episodes:  bias^2 of Q + rowΔ {float(((mean / len(ys) - gbar) ** 2).sum()):.1e} (noise / N = {nr / len(ys):.1e}),  "
      f"Var: Q {vq / len(ys):.4f} (exact {nq:.4f}), Q + rowΔ {vr / len(ys):.4f} (exact {nr:.4f})")

In [ ]:
M = 2000
rng = np.random.default_rng(0); total = collections.Counter()
for t in TASKS:
    tree = build_tree(t); f = leaf_rewards(tree, REWARD[t])
    seen = Seen()
    for y in sample(tree, M, rng):
        seen.add(tree.conf[y], float(f[y]))
    total["REINFORCE"] += noise(tree, torch.zeros_like(tree.P), f)                      # no baseline: the 10 reference
    for name, g in (("exact", f), ("fitted", leaf_rewards(tree, seen.f_hat()))):     # g: the reward table the control variate is built from
        Q = Q_table(tree, g)
        total[f"Q ({name})"] += noise(tree, Q, f)
        total[f"Q + Δ ({name})"] += noise(tree, Q + Delta_table(tree, g, Q), f)
        total[f"Q + rowΔ ({name})"] += noise_row(tree, Q, f, increments(tree, Q, g))
    del tree

share = {k: 10 * v / total["REINFORCE"] for k, v in total.items()}
names = list(share)[::-1]
plt.figure(figsize=(6, 3.5))
plt.barh(names, [share[k] for k in names], color=["C0" if "rowΔ" in k else "C9" if "Δ" in k else "C1" if k.startswith("Q") else "C3" for k in names])
for i, k in enumerate(names): plt.text(share[k] + 0.1, i, f"{share[k]:.2f}", va="center")
plt.xlabel("gradient noise (REINFORCE = 10)"); plt.xlim(0, max(share.values()) * 1.17); plt.tight_layout(); plt.show()
for name in ("exact", "fitted"):
    print(f"of the noise left by {name} Q, Δ removes {1 - share[f'Q + Δ ({name})'] / share[f'Q ({name})']:.0%} "
          f"and the row-wise residual removes {1 - share[f'Q + rowΔ ({name})'] / share[f'Q ({name})']:.0%}.")

In [ ]:
import sys; sys.path.insert(0, ".."); import torch.nn.functional as F
from baselines import group_weights, relax_weights, Surrogate
GROUP = 4                                                        # episodes per job per step for the group baselines
BASELINES = {"RLOO": GROUP, "GRPO": GROUP, "OTB": GROUP, "RELAX": 1}
SCORED = [k for k, t in enumerate(TEMPLATE) if t is None or FREE_TEXT]     # the positions the tree scores, as in build_tree
DIGIT_POS = [i for i, k in enumerate(SCORED) if TEMPLATE[k] is None]      # which of them write a digit

def decisions(tree, ys):                                         # each leaf as its sequence of decisions: one entry per position, -1 after an early end
    return torch.nn.utils.rnn.pad_sequence([tree.taken[y].nonzero().squeeze(1) for y in ys], batch_first=True, padding_value=-1)

def nodes(tree, ys, T=len(SCORED), A=5):                       # the nodes each leaf went through: their entries (K, T, A), -1 = none; the branch taken (K, T)
    ent = torch.full((len(ys), T, A), -1, device="cuda"); b = torch.zeros(len(ys), T, dtype=torch.long, device="cuda")
    for i, y in enumerate(ys):
        es = tree.passed[y].nonzero().squeeze(1)                 # every branch of every node on the way, in position order
        for t, n in enumerate(tree.node[es].unique()):
            g = es[tree.node[es] == n]; ent[i, t, :len(g)] = g; b[i, t] = (tree.taken[y][g] > 0).nonzero()
    return ent, b

def state(b, T=len(SCORED)):                                   # what RELAX's surrogate knows at each position: the position, and the digits already written
    t = torch.arange(T, device="cuda")
    digits = [F.one_hot(torch.where(t > k, b[:, k, None], 4), 5) for k in DIGIT_POS[:-1]]     # 4 = not written yet
    return torch.cat([F.one_hot(t, T).expand(len(b), T, T), *digits], -1).float()

def baseline_weights(method, tree, ys, f, surrogates, task, gen):     # w with ĝ = w @ Z, like weights(tree, c, f)[ys].mean(0)
    if method == "RELAX":
        ent, b = nodes(tree, ys); st = state(b)
        if task not in surrogates: surrogates[task] = Surrogate(st.shape[-1] + 5).cuda()
        c = surrogates[task]
        w = relax_weights(c, torch.log(tree.P.clamp_min(1e-300))[ent.clamp_min(0)].float(), ent, b, f[ys].float(), len(tree.P), st, gen)
        g = w.detach(); c.fit(w, tree.K)
        return g
    pos = decisions(tree, ys)
    return group_weights(method, f[ys], (pos >= 0).double(), pos, tree.K.diagonal())

### Is any of this estimating the right thing?

`REINFORCE` here is the plain estimator, $f\,\nabla \log p(y)$, with no baseline at all.  It sets the 10 on this scale.  A mean-reward baseline
that is known exactly would take that to 9.3, but in a run the constant has to be estimated from the run's own handful of
episodes, and then it costs more than it saves: the same runs with a mean-reward baseline are noisier over the first ten steps
and train worse (0.380 against 0.438 over the first 100 episodes).  The fitted `Q` table below is held back by the same thing,
one step further along.

RLOO subtracts the mean of the other three episodes in its group, and at P(success) around 0.16 most groups are all failures, so
that baseline is mostly noise: one RLOO episode is nearly twice as noisy as one REINFORCE episode.  GRPO divides by the group's
standard deviation and OTB's within-group baseline is not leave-one-out, so neither of them estimates the gradient of E[f] - the
middle column says so.  The group methods also spend four episodes per optimizer step, so at the same episode budget they take a
quarter of the Adam steps.

So REINFORCE is a strong baseline on this task.  The comparison that isolates the residual is `Q` against `Q + Δ` against
`Q + rowΔ`: the same episodes, the same steps, the same reward table.

In [ ]:
METHODS = ["REINFORCE", "Q", "Q + Δ", "Q + rowΔ", "Q (exact)", "Q + Δ (exact)", "Q + rowΔ (exact)"]   # "(exact)": the true reward in place of f-hat, a ceiling

tree = build_tree(job); f = leaf_rewards(tree, REWARD[job]); g = ((tree.pb * f) @ tree.taken).float() @ tree.Z     # the true gradient
g2 = float((g.double() ** 2).sum()); rel = lambda m: float(((m - g).double() ** 2).sum()) / g2
rf = noise(tree, torch.zeros_like(tree.P), f)                                    # plain REINFORCE, the = 10 reference
seen = Seen(); rng = np.random.default_rng(0)
for y in sample(tree, 2000, rng): seen.add(tree.conf[y], float(f[y]))            # enough episodes that f-hat has settled

print(f"{job}:" + " " * 18 + "|E[g_hat] - g|^2 / |g|^2                Var of one episode (REINFORCE = 10)")
for m in METHODS:
    c, xi = control_variate(m, tree, seen, f)
    mean = (tree.pb @ weights(tree, c, f)).float() @ tree.Z
    if xi is not None:                                                           # the row-wise residual is subtracted per episode
        mean = mean - tree.pb.float() @ torch.stack([row_correction(tree, xi, y) for y in range(len(tree.conf))]).float()
    v = noise(tree, c, f) if xi is None else noise_row(tree, c, f, xi)
    print(f"  {m:<18} {rel(mean):>8.1e}  exact{' ' * 21}{10 * v / rf:6.2f}")
gen = torch.Generator("cuda").manual_seed(0)
for m, K in BASELINES.items():                                                   # no table, so Monte Carlo over groups of K
    acc, acc2, sur, reps = 0.0, 0.0, {}, 3000 // K
    for _ in range(reps):
        gh = baseline_weights(m, tree, sample(tree, K, rng), f, sur, job, gen).float() @ tree.Z
        acc = acc + gh; acc2 += float((gh.double() ** 2).sum())
    mu = acc / reps; var = acc2 / reps - float((mu.double() ** 2).sum())
    print(f"  {m:<18} {rel(mu):>8.1e}  {reps} groups of {K}, so +-{var / reps / g2:.0e}   {10 * K * var / rf:6.2f}")

In [ ]:
TRAIN = ["port", "digest", "defines", "lint"]
EPISODES, STEPS, LR = 1, 60, 1e-3                              # episodes per job per step (4 per step in total)
SEEDS = list(range(20))        # three seeds cannot rank these: single runs either solve the task or collapse
RUNS = "_rowwise_runs.npz"     # ~3.5 h on one H100; delete the file to run it again, or split the seeds over several GPUs

def train(method, seed, episodes=EPISODES, steps=STEPS):
    reset(); rng = np.random.default_rng(seed); gen = torch.Generator("cuda").manual_seed(seed); opt = torch.optim.Adam(params, lr=LR)
    seen = {t: Seen() for t in TRAIN}; surrogates = {}; log = []; dead = False
    for step in range(steps + 1):
        grad, success, var, var_rf = 0.0, [], 0.0, 0.0
        for t in (TRAIN if not dead else []):
            tree = build_tree(t); f = leaf_rewards(tree, REWARD[t])
            if not torch.isfinite(tree.pb).all():                    # the step before it sent the weights to NaN: keep the run as a collapse
                print(f"  {method}, seed {seed}: the policy blew up at episode {step * episodes * len(TRAIN)}"); dead = True; del tree; break
            success.append(float(tree.pb @ f)); var_rf += noise(tree, torch.zeros_like(tree.P), f)      # plain REINFORCE, the = 1 reference
            if method in BASELINES: var = np.nan                     # no table, so no exact noise
            else: c, xi = control_variate(method, tree, seen[t], f)
            if step < steps:
                ys = sample(tree, episodes, rng)
                if method in BASELINES: g = baseline_weights(method, tree, ys, f, surrogates, t, gen).float() @ tree.Z
                else: g, v = estimator(tree, c, xi, f, ys); var += v
                grad = grad + g / len(TRAIN)
                for y in ys: seen[t].add(tree.conf[y], float(f[y]))
            elif method not in BASELINES: var += noise(tree, c, f) if xi is None else noise_row(tree, c, f, xi)
            del tree
        if dead: log.append((step * episodes * len(TRAIN), 0.0, np.nan)); continue
        log.append((step * episodes * len(TRAIN), np.mean(success), var / max(var_rf, 1e-30)))
        if step < steps:
            opt.zero_grad()
            for p, g in zip(params, (-grad).split([p.numel() for p in params])): p.grad = g.view_as(p)
            opt.step()
    return np.array(log)

if os.path.exists(RUNS):
    z = np.load(RUNS); logs = {str(n): z[f"L{i}"] for i, n in enumerate(z["names"])}
    print(f"{RUNS}: {len(logs)} methods x {len(z['seeds'])} seeds")
else:
    logs = {m: np.stack([train(m, s) for s in SEEDS]) for m in METHODS}
    for m, K in BASELINES.items():                             # the same episode budget: K episodes per job per step, STEPS / K steps
        logs[m] = np.stack([train(m, s, episodes=K, steps=STEPS * EPISODES // K) for s in SEEDS])
    np.savez(RUNS, names=np.array(list(logs)), seeds=np.array(SEEDS), **{f"L{i}": L for i, L in enumerate(logs.values())})

In [ ]:
style = {"REINFORCE": ("C3", "-"), "Q": ("C1", "-"), "Q + Δ": ("C9", "-"), "Q + rowΔ": ("C0", "-"),
         "Q (exact)": ("C1", "--"), "Q + Δ (exact)": ("C9", "--"), "Q + rowΔ (exact)": ("C0", "--"),
         "RLOO": ("C7", "-"), "GRPO": ("C7", "--"), "OTB": ("C7", ":"), "RELAX": ("C4", "-")}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m, L in logs.items():                                      # a log row: (episodes so far, P(success), noise / REINFORCE's)
    col, ls = style.get(m, ("k", ":")); episodes = L[0, :, 0]
    ax[0].plot(episodes, L[:, :, 1].mean(0), color=col, ls=ls, lw=2, label=m)      # REINFORCE: no baseline
    if m in ("Q", "Q + rowΔ"):                                 # the individual runs of the pair the residual is meant to separate
        for run in L: ax[0].plot(episodes, run[:, 1], color=col, ls=ls, lw=0.5, alpha=0.15)
    if m not in BASELINES: ax[1].plot(episodes, np.exp(np.log(L[:, :, 2]).mean(0)), color=col, ls=ls, lw=2, label=m)
ax[0].set(xlabel="agent episodes", ylabel=f"P(success), exact, mean of {len(SEEDS)} seeds", ylim=(0, 1)); ax[0].legend(fontsize=8)
ax[1].set(xlabel="agent episodes", ylabel="noise / REINFORCE's, exact", yscale="log")
plt.tight_layout(); plt.show()

m100 = lambda L: L[:, L[0, :, 0] <= 100, 1].mean(1)            # one number per run: P(success) over the first 100 episodes
for m, L in logs.items():
    mean, fin = m100(L), L[:, -1, 1]
    print(f"{m:<17} solves {int((L[:, :, 1] >= 0.8).any(1).sum()):>2}/{len(L)}  collapses {int((fin < 0.05).sum()):>2}/{len(L)}   "
          f"first 100 episodes {mean.mean():.3f} +- {mean.std(ddof=1) / len(mean) ** 0.5:.3f}   final {fin.mean():.3f} +- {fin.std(ddof=1) / len(fin) ** 0.5:.3f}"
          f"   noise over the first 10 steps {np.exp(np.log(L[:, :10, 2]).mean()):.2f}")

from scipy import stats                                        # paired over the seeds: every method saw the same 20 seeds
print(f"\npaired difference in P(success) over the first 100 episodes:")
for a, b in [("Q + Δ", "Q"), ("Q + rowΔ", "Q"), ("Q + rowΔ", "Q + Δ"), ("Q", "REINFORCE"),
             ("Q + Δ (exact)", "Q (exact)"), ("Q + rowΔ (exact)", "Q (exact)"), ("Q + rowΔ (exact)", "Q + Δ (exact)")]:
    d = m100(logs[a]) - m100(logs[b])
    print(f"  {a:<18} - {b:<18} {d.mean():+.3f} +- {d.std(ddof=1) / len(d) ** 0.5:.3f}   "
          f"p {stats.ttest_rel(m100(logs[a]), m100(logs[b])).pvalue:.3f}   {int((d > 0).sum())}/{len(d)} seeds")